In [1]:
import os
os.chdir("..")

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from optimizer import grid_search, strategy_compare

In [3]:
SEARCH_SPACE = {
    "cooldown_long": [30, 60, 90],
    "cooldown_short": [7, 14, 30],
}

In [4]:
df_prices = pd.read_csv("data/test_crypto.csv", index_col=0, parse_dates=True)
df_prices

,BTC,ETH,SOL,XRP,ADA,DOT,LINK,XLM,XMR
Date,,,,,,,,,
2014-09-17,457.334015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-18,424.440002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-19,394.795990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-20,408.903992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-21,398.821014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-10-19,81853.539062,2404.815186,104.926231,1.302713,0.217995,1.153126,12.043903,0.177571,605.426392
2026-10-20,82941.000000,2446.730713,106.422340,1.334452,0.223782,1.149771,12.320927,0.181960,600.929504
2026-10-21,84011.968750,2470.766113,106.793655,1.363619,0.224979,1.141323,12.484719,0.183488,622.434082


# SEARCH

In [5]:
history_search = grid_search(
    prices=df_prices, 
    param_grid=SEARCH_SPACE, 
    n_boot=100,
    sample_size=10,
    sequence_length=365 * 4,
    )

100%|██████████| 9/9 [07:04<00:00, 47.20s/it]


In [6]:
history_search

,strategy,Target,Cash_spent,Realized_cash,Portfolio,Invested,Total_value,Bull_history,Avg_price,Num_take_profits,MDD,MA200_survival_days,RoAC,iteration
0,cooldown_long=30_cooldown_short=7,XLM,2574,2606,1334,1720,3940,43,0.115399,5,-528,33.692308,65,1
1,cooldown_long=30_cooldown_short=7,XRP,2316,5134,3860,618,8995,70,0.524709,2,-1254,89.000000,10,1
2,cooldown_long=30_cooldown_short=7,BTC,2069,4734,304,263,5038,73,25851.374359,7,-572,170.500000,68,1
3,cooldown_long=30_cooldown_short=7,BTC,2059,5706,211,139,5918,75,23466.267502,8,-971,119.750000,61,1
4,cooldown_long=30_cooldown_short=7,XLM,2841,7522,451,262,7974,33,0.153395,4,-2503,83.444444,-12,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8995,cooldown_long=90_cooldown_short=30,BTC,2316,6164,1036,299,7201,70,17800.399374,5,-1011,137.500000,75,100
8996,cooldown_long=90_cooldown_short=30,XMR,2069,1699,1221,1352,2920,62,160.422679,3,-809,47.500000,57,100
8997,cooldown_long=90_cooldown_short=30,LINK,2059,6423,276,428,6699,84,17.036022,10,-1476,0.000000,263,100
8998,cooldown_long=90_cooldown_short=30,SOL,2326,5172,617,890,5790,74,116.360908,6,-1037,0.000000,185,100


In [ ]:
search_res = history_search.groupby("strategy").agg(
    RoAC_mean=("RoAC", "mean"),
    RoAC_median=("RoAC", "median"),
    MDD_mean=("MDD", "mean"),
    MDD_median=("MDD", "median"),
    Portfolio_mean=("Portfolio", "mean"),
    Portfolio_median=("Portfolio", "median"),
    Cash_spent_mean=("Cash_spent", "mean"),
    Cash_spent_median=("Cash_spent", "median"),
    Total_value_mean=("Total_value", "mean"),
    Total_value_median=("Total_value", "median"),
    Total_value_low=("Total_value", lambda s: np.percentile(s, 25)),
    Total_value_high=("Total_value", lambda s: np.percentile(s, 75)),
)

search_res['Uncle'] = search_res['MDD_mean'] / search_res['Total_value_mean']
search_res['Profit_coef'] = search_res['Total_value_mean'] / search_res['Cash_spent_mean']
search_res.sort_values("RoAC_mean", ascending=False).style.background_gradient(cmap="RdYlGn", axis=0).format("{:.2f}")

,RoAC,MDD,Portfolio,Realized_cash,Cash_spent,Total_value_mean,Total_value_meadian,Total_value_low,Total_value_high
strategy,,,,,,,,,
cooldown_long=90_cooldown_short=30,77.52,-1552.53,1278.51,4241.92,2343.55,5520.89,4925.50,3732.75,7026.50
cooldown_long=90_cooldown_short=14,71.49,-1441.29,1211.30,4113.74,2366.05,5325.52,4703.50,3499.25,6484.75
cooldown_long=90_cooldown_short=7,68.63,-1441.87,1166.81,4074.03,2385.53,5241.31,4788.00,3590.25,6505.50
cooldown_long=60_cooldown_short=14,67.45,-1145.83,1081.96,4088.83,2419.85,5171.25,4856.00,3763.25,6509.00
cooldown_long=60_cooldown_short=30,66.15,-1283.90,1257.26,4000.91,2410.32,5258.61,4689.00,3635.75,6777.50
cooldown_long=60_cooldown_short=7,64.28,-1093.58,1109.83,3827.35,2421.05,4937.65,4579.00,3577.75,6173.50
cooldown_long=30_cooldown_short=30,62.56,-1125.83,1151.06,3777.90,2438.31,4929.40,4652.50,3617.25,6113.25
cooldown_long=30_cooldown_short=14,58.07,-988.21,1065.35,3622.30,2433.92,4688.08,4498.50,3673.00,5719.25
cooldown_long=30_cooldown_short=7,51.80,-999.20,1068.95,3452.22,2450.17,4521.61,4352.50,3486.25,5503.50
